In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Using GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16 if DEVICE == "cuda" else torch.float32
).to(DEVICE)

model.eval()

print("Model device:", next(model.parameters()).device)
print("Model loaded successfully.")

c:\Users\VINIL\Desktop\Failure_Aware_Agent\Environment\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA available: True
Using GPU: NVIDIA GeForce RTX 4060 Laptop GPU


Loading weights: 100%|██████████| 434/434 [00:04<00:00, 96.75it/s, Materializing param=model.norm.weight]                               


Model device: cuda:0
Model loaded successfully.


In [4]:
# ==============================
# INSTALL FIRST
# ==============================
# pip install datasets tqdm radon sentence-transformers faiss-cpu pandas numpy torch

import ast
import json
import re
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from datasets import load_dataset
from radon.complexity import cc_visit
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer
import faiss
import os

torch.set_grad_enabled(False)

# ==============================
# CONFIGURATION
# ==============================
MAX_SAMPLES = 10
MAX_PROMPT_LEN = 400
MAX_NEW_TOKENS = 4096
TOP_K_HISTORY = 2

CSV_FILE = "failure_risk_dataset.csv"
FAISS_FILE = "history_index.faiss"
LABEL_FILE = "history_labels.npy"

# ==============================
# LOAD DATASET AND MODELS
# ==============================
dataset = load_dataset("code_search_net", "python")["train"]
dataset = dataset.select(range(min(MAX_SAMPLES, len(dataset))))

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embedding_dim = 384

# Language model (assumed to be loaded elsewhere)
# Make sure tokenizer, model, DEVICE are defined.

# ==============================
# LOAD OR CREATE VECTOR DB
# ==============================
if os.path.exists(FAISS_FILE):
    faiss_index = faiss.read_index(FAISS_FILE)
    history_labels = list(np.load(LABEL_FILE))
else:
    faiss_index = faiss.IndexFlatL2(embedding_dim)
    history_labels = []

# ==============================
# SEED HISTORY WITH A SUCCESS (if empty)
# ==============================
if len(history_labels) == 0:
    seed_code = "def add(a, b):\n    return a + b"
    seed_prompt = "Write a function that adds two numbers."
    seed_embedding = embedding_model.encode(seed_prompt + " " + seed_code)
    faiss_index.add(np.array([seed_embedding]).astype("float32"))
    history_labels.append(0)
    print("Seeded history with a successful sample.")

# ==============================
# UTILITY FUNCTIONS
# ==============================
def clean_code(code: str) -> str:
    """Remove markdown fences and strip whitespace."""
    code = code.replace("```python", "").replace("```", "").strip()
    return code

def extract_json(text: str):
    """Find the first '{' and last '}' and attempt to parse JSON."""
    start = text.find('{')
    end = text.rfind('}')
    if start != -1 and end != -1 and end > start:
        json_str = text[start:end+1]
        try:
            return json.loads(json_str)
        except json.JSONDecodeError:
            return None
    return None

def extract_any_python_code(text: str) -> str:
    """
    Most aggressive fallback: look for any line that starts with Python keywords
    or looks like Python code, and collect until a blank line or end.
    """
    lines = text.split('\n')
    code_lines = []
    in_code_block = False
    
    python_start_patterns = [
        r'^\s*def\s+\w+\s*\(',           # function definition
        r'^\s*class\s+\w+',               # class definition
        r'^\s*import\s+\w+',               # import statement
        r'^\s*from\s+\w+\s+import',        # from import
        r'^\s*@\w+',                       # decorator
        r'^\s*if\s+.*:',                    # if statement
        r'^\s*for\s+.*:',                   # for loop
        r'^\s*while\s+.*:',                 # while loop
        r'^\s*return\s+',                   # return statement
        r'^\s*print\s*\(',                  # print function
        r'^\s*[a-zA-Z_]\w*\s*=\s*',         # assignment
        r'^\s*[a-zA-Z_]\w*\s*\(',           # function call
    ]
    
    i = 0
    while i < len(lines):
        line = lines[i]
        
        # Check if this line looks like Python code
        looks_like_python = False
        for pattern in python_start_patterns:
            if re.match(pattern, line):
                looks_like_python = True
                break
        
        if looks_like_python and not in_code_block:
            in_code_block = True
            code_lines.append(line)
        elif in_code_block:
            # If line is empty or doesn't look like Python, maybe end of block
            if line.strip() == '':
                # Empty line might be end of function, but could be inside
                # Check next line to decide
                if i + 1 < len(lines):
                    next_line = lines[i + 1]
                    next_looks_like_python = False
                    for pattern in python_start_patterns:
                        if re.match(pattern, next_line):
                            next_looks_like_python = True
                            break
                    if not next_looks_like_python:
                        break
                code_lines.append(line)
            else:
                # Check if still indented (continuation)
                current_indent = len(line) - len(line.lstrip())
                if current_indent > 0 or line.strip() == '':
                    code_lines.append(line)
                else:
                    break
        i += 1
    
    if code_lines:
        return '\n'.join(code_lines).strip()
    return ""

def extract_code_from_text(text: str) -> str:
    """
    Multi-stage fallback to extract Python code from raw text.
    """
    # Debug: print raw text for first few samples
    if not hasattr(extract_code_from_text, "counter"):
        extract_code_from_text.counter = 0
    if extract_code_from_text.counter < 3:
        print(f"\n--- RAW MODEL OUTPUT (Sample {extract_code_from_text.counter}) ---")
        print(text[:500] + "..." if len(text) > 500 else text)
        print("--- END RAW OUTPUT ---\n")
        extract_code_from_text.counter += 1
    
    # 1. Look for JSON first
    data = extract_json(text)
    if data and "code" in data:
        return data["code"].strip()
    
    # 2. Look for code in markdown blocks
    pattern = r"```(?:python)?\n(.*?)```"
    matches = re.findall(pattern, text, re.DOTALL)
    if matches:
        return matches[-1].strip()
    
    # 3. Look for any Python-like content
    python_code = extract_any_python_code(text)
    if python_code:
        return python_code
    
    return ""

def generate_reasoning_code(prompt: str, func_name: str):
    """
    Call the language model to produce reasoning and code.
    Returns (reasoning, code, mean_logprob).
    """
    prompt = prompt[:MAX_PROMPT_LEN]

    # Simplified prompt - just ask for code directly
    full_prompt = f"""Write a Python function named '{func_name}' that solves this problem:
{prompt}

Return ONLY the Python code, no explanations, no markdown, just the function definition."""

    inputs = tokenizer(full_prompt, return_tensors="pt").to(DEVICE)

    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        return_dict_in_generate=True,
        output_scores=True,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_tokens = outputs.sequences[0]
    new_tokens = generated_tokens[inputs.input_ids.shape[1]:]

    logprobs = []
    for i, score in enumerate(outputs.scores):
        probs = F.log_softmax(score, dim=-1)
        logprobs.append(probs[0, new_tokens[i]].item())
    mean_logprob = float(np.mean(logprobs)) if logprobs else -100

    text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    
    # Extract code using our robust function
    code = extract_code_from_text(text)
    
    return "", code, mean_logprob  # reasoning is empty now

def compute_confidence(mean_logprob: float) -> float:
    return float(np.exp(mean_logprob))

def check_syntax(code: str) -> int:
    """Return 1 if code has a syntax error, else 0."""
    if not code.strip():
        return 1
    try:
        compile(code, "<string>", "exec")
        return 0
    except SyntaxError as e:
        # Print syntax error for debugging
        if not hasattr(check_syntax, "error_counter"):
            check_syntax.error_counter = 0
        if check_syntax.error_counter < 3:
            print(f"\nSyntax error in code: {e}")
            print("Code that caused error:")
            print(code[:200] + "..." if len(code) > 200 else code)
            check_syntax.error_counter += 1
        return 1

def compute_prior_history(embedding: np.ndarray) -> float:
    """Compute prior failure probability from nearest neighbors."""
    if len(history_labels) < TOP_K_HISTORY:
        return 0.5
    query = np.array([embedding]).astype("float32")
    distances, indices = faiss_index.search(query, TOP_K_HISTORY)
    similarities = 1.0 / (1.0 + distances[0])
    neighbor_labels = np.array(history_labels)[indices[0]]
    prior = np.sum(similarities * neighbor_labels) / np.sum(similarities)
    return float(prior)

def extract_features(prompt: str, code: str, mean_logprob: float, prior: float):
    """Compute all features for a sample."""
    features = {}
    features["prompt_len"] = len(prompt)
    features["code_len"] = len(code)

    # AST node count (only if code is valid)
    try:
        tree = ast.parse(code)
        features["ast_nodes"] = sum(1 for _ in ast.walk(tree))
    except SyntaxError:
        features["ast_nodes"] = 0

    # Cyclomatic complexity (only if code is valid)
    try:
        complexity = cc_visit(code)
        features["avg_complexity"] = (
            sum(c.complexity for c in complexity) / len(complexity) if complexity else 0
        )
    except Exception:
        features["avg_complexity"] = 0

    features["mean_logprob"] = mean_logprob
    features["confidence"] = compute_confidence(mean_logprob)
    features["prior_history"] = prior
    return features

# ==============================
# CSV INIT
# ==============================
if not os.path.exists(CSV_FILE):
    cols = [
        "prompt_len",
        "code_len",
        "ast_nodes",
        "avg_complexity",
        "mean_logprob",
        "confidence",
        "prior_history",
        "failed"
    ]
    pd.DataFrame(columns=cols).to_csv(CSV_FILE, index=False)

# ==============================
# MAIN LOOP
# ==============================
skipped = 0
for i, item in enumerate(tqdm(dataset)):
    prompt = item["func_documentation_string"]
    if len(prompt) > 600:
        skipped += 1
        continue

    reference_code = item["func_code_string"]
    try:
        func_name = reference_code.split("def ")[1].split("(")[0]
    except IndexError:
        func_name = "solution"

    try:
        reasoning, code, mean_logprob = generate_reasoning_code(prompt, func_name)
    except Exception as e:
        print(f"Generation failed for sample {i}: {e}")
        skipped += 1
        continue

    label = check_syntax(code)

    # Combine prompt and reasoning for embedding
    embedding = embedding_model.encode(prompt + " " + reasoning)
    prior = compute_prior_history(embedding)

    feats = extract_features(prompt, code, mean_logprob, prior)
    feats["failed"] = label

    # Append to CSV
    pd.DataFrame([feats]).to_csv(CSV_FILE, mode="a", header=False, index=False)

    # Update history
    history_labels.append(label)
    faiss_index.add(np.array([embedding]).astype("float32"))

    # Periodically save index and labels
    if (i + 1) % 10 == 0:
        faiss.write_index(faiss_index, FAISS_FILE)
        np.save(LABEL_FILE, np.array(history_labels))

    # Debug: print first few samples
    if i < 3:
        print(f"\n--- Sample {i} ---")
        print(f"Prompt: {prompt[:100]}...")
        print(f"Extracted code length: {len(code)}")
        print(f"Extracted code preview:\n{code[:200]}...")
        print(f"Syntax OK? {label == 0}")
        print(f"Prior: {prior:.4f}")

# Final save
faiss.write_index(faiss_index, FAISS_FILE)
np.save(LABEL_FILE, np.array(history_labels))

print("Dataset generation finished")
print("Skipped:", skipped)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 885.95it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Seeded history with a successful sample.


 10%|█         | 1/10 [00:12<01:49, 12.21s/it]


--- RAW MODEL OUTPUT (Sample 0) ---
 ```python
import numpy as np
from scipy.ndimage import filters

def __msgc_step3_discontinuity_localization(image):
    """
    Estimate discontinuity in basis of low resolution image segmentation.

    :param image: 2D numpy array representing the low resolution image.
    :return: Discontinuity in low resolution image.
    """
    # Compute the gradient magnitude of the image
    gradient_magnitude = np.hypot(filters.gaussian_filter(image, sigma=1),
                                   filters....
--- END RAW OUTPUT ---


--- Sample 0 ---
Prompt: Estimate discontinuity in basis of low resolution image segmentation.
        :return: discontinuity...
Extracted code length: 832
Extracted code preview:
import numpy as np
from scipy.ndimage import filters

def __msgc_step3_discontinuity_localization(image):
    """
    Estimate discontinuity in basis of low resolution image segmentation.

    :param ...
Syntax OK? True
Prior: 0.5000


 20%|██        | 2/10 [00:27<01:51, 13.91s/it]


--- RAW MODEL OUTPUT (Sample 1) ---
 ```python
def __multiscale_gc_lo2hi_run(low_res_graph, low_res_labels, high_res_graph, high_res_labels, boundary_penalties):
    import numpy as np
    from scipy.sparse.csgraph import connected_components

    # Perform initial graph cut segmentation at low resolution
    labels = connected_components(low_res_graph, connection='strong')
    
    # Refine the segmentation using the high-resolution graph and boundary penalties
    refined_labels = np.zeros_like(high_res_labels)
    for i in rang...
--- END RAW OUTPUT ---


--- Sample 1 ---
Prompt: Run Graph-Cut segmentation with refinement of low resolution multiscale graph.
        In first step...
Extracted code length: 1137
Extracted code preview:
def __multiscale_gc_lo2hi_run(low_res_graph, low_res_labels, high_res_graph, high_res_labels, boundary_penalties):
    import numpy as np
    from scipy.sparse.csgraph import connected_components

   ...
Syntax OK? True
Prior: 0.0000


 30%|███       | 3/10 [00:42<01:41, 14.47s/it]


--- RAW MODEL OUTPUT (Sample 2) ---
 ```python
def __multiscale_gc_hi2lo_run(low_res_data, high_res_data, boundary_penalties, **kwargs):
    import numpy as np
    from skimage.segmentation import slic
    from skimage.segmentation import mark_boundaries
    from skimage.util import img_as_float
    from scipy.sparse.csgraph import minimum_spanning_tree

    # Convert to float and normalize
    low_res_data = img_as_float(low_res_data)
    high_res_data = img_as_float(high_res_data)

    # Perform initial graph cut segmentation at...
--- END RAW OUTPUT ---


--- Sample 2 ---
Prompt: Run Graph-Cut segmentation with simplifiyng of high resolution multiscale graph.
        In first st...
Extracted code length: 1212
Extracted code preview:
def __multiscale_gc_hi2lo_run(low_res_data, high_res_data, boundary_penalties, **kwargs):
    import numpy as np
    from skimage.segmentation import slic
    from skimage.segmentation import mark_bou...
Syntax OK? True
Prior: 0.0000


100%|██████████| 10/10 [02:35<00:00, 15.54s/it]

Dataset generation finished
Skipped: 0


In [4]:
import torch
torch.cuda.empty_cache()

In [2]:
!nvidia-smi

Fri Mar  6 14:17:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 555.97                 Driver Version: 555.97         CUDA Version: 12.5     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   54C    P0             14W /   87W |       0MiB /   8188MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----